In [1]:
!pip install nltk rouge-score evaluate

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25024 sha256=d6a56437e6f116f463f45abf63bbb15044e1732700253465f7aa7138096c8353
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\1e\19\43\8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score

   ---------------------------------------- 0/3 [absl-py]
   ---------------------------------------- 0/3 [absl-py]
   ---------------------------------------- 0/3 [absl-py]
   ---------------------------------------- 0/3 [absl-py]
   ---------------------------------------- 0/3 [absl-py]
   ---------------------------------------- 0/3 [absl-py]


In [2]:
import os
import random
import pandas as pd

from PIL import Image

from transformers import BlipProcessor, BlipForConditionalGeneration

import nltk
from nltk.translate.bleu_score import sentence_bleu

from rouge_score import rouge_scorer

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...


True

In [3]:
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [5]:
dataset_path = r"C:\Users\Lenovo\Internship\data\Flickr8k"

image_folder = os.path.join(dataset_path, "images")

caption_file = os.path.join(dataset_path, "captions.txt")

In [6]:
captions = pd.read_csv(
    caption_file,
    sep="|"
)

captions.head()

,image_name,caption_number,caption_text
0,1000268201_693b08cb0e.jpg,0,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,1,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,2,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,3,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,4,A little girl in a pink dress going into a woo...


In [7]:
image_names = captions["image_name"].unique()

validation_images = random.sample(
    list(image_names),
    200
)

In [10]:
generated = []
reference = []

for img in validation_images:

    image = Image.open(
        os.path.join(image_folder, img)
    ).convert("RGB")

    inputs = processor(images=image, return_tensors="pt")

    output = model.generate(**inputs)

    prediction = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    generated.append(prediction)

    gt = captions[
        captions["image_name"] == img
    ]["caption_text"].iloc[0]

    reference.append(gt)

c:\Users\Lenovo\Internship\venv\Lib\site-packages\transformers\generation\utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [12]:
bleu_scores = []

for ref, pred in zip(reference, generated):

    score = sentence_bleu(
        [ref.split()],
        pred.split()
    )

    bleu_scores.append(score)

average_bleu = sum(bleu_scores) / len(bleu_scores)

print("Average BLEU:", average_bleu)

Average BLEU: 0.010374505531065314


c:\Users\Lenovo\Internship\venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Lenovo\Internship\venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Lenovo\Internship\venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use Smoothi

In [13]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1','rougeL'],
    use_stemmer=True
)

rouge1=[]
rougel=[]

for ref,pred in zip(reference,generated):

    score = scorer.score(ref,pred)

    rouge1.append(score["rouge1"].fmeasure)

    rougel.append(score["rougeL"].fmeasure)

print("ROUGE-1:",sum(rouge1)/len(rouge1))

print("ROUGE-L:",sum(rougel)/len(rougel))

ROUGE-1: 0.3907296221782246
ROUGE-L: 0.37321807426508746


In [14]:
from nltk.translate.meteor_score import meteor_score

meteor=[]

for ref,pred in zip(reference,generated):

    meteor.append(
        meteor_score(
            [ref.split()],
            pred.split()
        )
    )

print("METEOR:",sum(meteor)/len(meteor))

METEOR: 0.22644682729387822


In [15]:
results = pd.DataFrame({

    "Metric":[
        "BLEU",
        "ROUGE-1",
        "ROUGE-L",
        "METEOR"
    ],

    "Score":[
        average_bleu,
        sum(rouge1)/len(rouge1),
        sum(rougel)/len(rougel),
        sum(meteor)/len(meteor)
    ]

})

results

,Metric,Score
0,BLEU,0.010375
1,ROUGE-1,0.390730
2,ROUGE-L,0.373218
3,METEOR,0.226447


## Baseline Evaluation

The pretrained BLIP model was evaluated on a validation subset of 200 Flickr8k images.

BLEU, ROUGE, and METEOR metrics were calculated by comparing generated captions with the ground-truth captions.

The model correctly identified the main objects and actions in many images, but some captions lacked detailed contextual information. These scores provide a baseline for comparison with future fine-tuned models.